# Web Scraping Viva Real

## Importando bibliotecas

In [1]:
import re, time, os
import numpy as np
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
import pandas as pd
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

## Web Scraper

In [14]:
# Inicializamos as listas para guardar as informações
link_imovel = []  # nesta lista iremos guardar a url
id_list = []      # nesta lista iremos guardar o id (renamed to avoid conflict with built-in 'id')
address = []      # nesta lista iremos guardar o endereço
anunciante = []   # nesta lista iremos guardar o anunciante 
area = []         # nesta lista iremos guardar a area
price = []        # nesta lista iremos guardar o preço do imóvel
rooms = []        # nesta lista iremos guardar o número de quartos

# Carregar IDs existentes para evitar duplicatas
existing_ids = set()
csv_file = 'DfImoveis.csv'
if os.path.exists(csv_file):
    try:
        existing_df = pd.read_csv(csv_file, encoding='utf-16', header=None)  # Ajuste encoding se necessário
        existing_ids = set(existing_df[0].astype(str))  # Coluna 0 é o ID
        print(f"Loaded {len(existing_ids)} existing IDs from {csv_file}")
    except Exception as e:
        print(f"Error loading existing CSV: {e}. Starting fresh.")

# Ele irá solicitar qual página você deseja coletar
page_number = int(input('Qual página deseja coletar? '))
# inicializa o tempo de execução
tic = time.time()

# Configure chromedriver
chromedriver = "./chromedriver.exe"
service = Service(chromedriver)
options = Options()
options.binary_location = "C:\\Program Files\\Google\\Chrome Beta\\Application\\chrome.exe"  # caminho do seu navegador Chrome
driver = webdriver.Chrome(service=service, options=options)
actions = ActionChains(driver)

# Informe o link inicial para inicializar o scraping. Você pode trocar entre diversos filtros.
link = f'https://www.dfimoveis.com.br/venda/df/aguas-claras/apartamento?&ordenamento=mais-recente'

# navegamos até a página desejada
driver.get(f"{link}&pagina={page_number}")
# coletamos todas as informações da página e transformamos em formato legivel
data = driver.execute_script("return document.getElementsByTagName('html')[0].innerHTML")
soup_complete_source = BeautifulSoup(data.encode('utf-8'), 'html.parser')

# identificamos todos os itens de card de imóveis
soup = soup_complete_source.find(id='resultadoDaBuscaDeImoveis')    

# Web-Scraping
# para cada elemento no conjunto de cards, colete:
for line in soup.find_all(class_="imovel-card"):
    try:
        # Coleta o link
        full_link = line.get('href')
        
        # Coleta o id do imóvel
        property_id = full_link.split('-')[-1]
        
        # Verifica se o ID já existe (evita duplicatas)
        if property_id in existing_ids:
            continue  # Pula este imóvel

        #Adicionar Quartos
        for pilula in line.find_all(class_="border-1 py-0 px-2 bg-white body-small rounded-pill"):
            if 'Quarto' in pilula.text or 'Quartos' in pilula.text:
                rooms.append(pilula.text.strip())
                break
            else:
                rooms.append('N/A')  # Caso não encontre a informação de quartos
        
        # Adiciona aos listas apenas se for novo
        link_imovel.append(f'https://www.dfimoveis.com.br{full_link}')
        id_list.append(property_id)
        existing_ids.add(property_id)  # Adiciona ao set para futuras verificações
        
        full_address = line.find(class_="ellipse-text body-medium accent-color bold").text.strip()
        address.append(full_address.replace('\n', ''))
                        
        # Coleta o anunciante
        full_anunciante = line.find(class_='imovel-anunciante').picture.img.get('alt').title()
        anunciante.append(full_anunciante)
                
        # Coleta a área  
        full_area = line.find(class_="ellipse-text body-small accent-color bold mobile-ellipse-view").text.strip()
        areas = re.findall(r'\d+', full_area)
        areas = [int(i) for i in areas]
        area.append(np.mean(areas))  # Get apto's area
        
        # Coleta preço
        full_price = re.sub('[^0-9]', '', line.css.select(".imovel-price > h4 > span")[0].text.strip())
        price.append(full_price) 

    except Exception as e:
        print(f"Error processing a property: {e}")
        continue
         
# fecha o chromedriver
driver.quit()

# Cria um dataframe pandas e salva como um arquivo CSV (apenas novos registros)
if id_list:  # Só salva se houver novos dados
    new_data = pd.DataFrame({
        'ID': id_list,
        'Link': link_imovel,
        'Address': address,
        'Anunciante': anunciante,
        'Area': area,
        'Price': price,
        'Rooms': rooms
    })
    
    # Se o arquivo já existe, anexa sem cabeçalho; senão, inclui cabeçalho
    mode = 'a' if os.path.exists(csv_file) else 'w'
    header = not os.path.exists(csv_file)
    new_data.to_csv(csv_file, mode=mode, header=header, index=False, encoding='utf-16')
    print(f"Added {len(new_data)} new records to {csv_file}")
else:
    print("No new records to add.")

# Tempo de execução
toc = time.time()
get_time = round(toc - tic, 3)
print('Finished in ' + str(get_time) + ' seconds')
print(str(len(price)) + ' new results!')

Loaded 31 existing IDs from DfImoveis.csv
Added 30 new records to DfImoveis.csv
Finished in 5.018 seconds
30 new results!
